# 13 · In-memory, on-disk and server modes
Level L2 · Part 2 · builds on lesson 12 (and lessons 07-11)

Needs: nothing for the main path. The `needs-docker` cells start one Qdrant server
(`qdrant/qdrant:v1.19.2`, about 1 GB RAM, ports 6333 and 6334 must be free). When they are
skipped, the lesson prints the **recorded output** of a real run from `data/recorded/13.json`.

## Goal
By the end of this lesson you can run the same recipe search against an in-memory database, a
folder on disk, a Qdrant server in Docker and Qdrant Edge, and say which one to use for what.

## The idea
Lessons 07-12 used **local mode**: `QdrantClient(path=...)` runs a small Python copy of Qdrant
inside your notebook. It is great for learning and tests, but it scans every vector, builds no
index and quietly ignores search tuning. A **Qdrant server** is the real engine: a separate
program you talk to over the network, with real indexes, many clients at once, and data that
outlives your notebook. **Qdrant Edge** is that same engine as a Python library for offline apps
on one device. The client code barely changes, so you develop on one mode and ship on another. Think of a practice kitchen at home, a restaurant
kitchen, and a food truck: the recipes are the same, the equipment is not.

## Picture

```mermaid
flowchart LR
    NB["your notebook<br/>build_recipes() + ask()"]
    NB -->|"QdrantClient(':memory:')"| M["local mode, RAM<br/>gone when closed"]
    NB -->|"QdrantClient(path=...)"| F["local mode, folder<br/>survives restart"]
    NB -->|"QdrantClient(url=...)<br/>HTTP :6333"| S["Qdrant server (Docker)<br/>real indexes, many clients"]
    NB -->|"EdgeShard (other API)"| E["Qdrant Edge<br/>real engine in-process"]
```

## Step by step
### Step 1 · The recipes, and two functions that take any client
Same data as lesson 12: 50 recipes and their saved MiniLM vectors, plus the cached query "something
warm for a cold day".

In [1]:
import json, os, shutil, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="IProgress")     # quiet a progress-bar notice on import
from qdrant_client import QdrantClient, models

recipes = pd.read_csv("../data/recipes_50.csv")
saved = np.load("../data/recipe_embeddings_minilm.npz")
doc_emb, MODEL = saved["doc_emb"], str(saved["model"])
cache = np.load("../data/query_embeddings_minilm.npz")
warm_query = cache["query_emb"][0].tolist()
print("model:", MODEL, "| recipes:", doc_emb.shape, "| query:", str(cache["queries"][0]))

model: sentence-transformers/all-MiniLM-L6-v2 | recipes: (50, 384) | query: something warm for a cold day


The point of this lesson is that **this code does not know where the data lives**. Both functions
take a `client` argument: `build_recipes` (re)creates the collection, `ask` runs the search for
quick recipes only (`minutes <= 30`, lesson 11) and returns the top 3 as `[title, score]`.

In [2]:
def to_payload(row):
    return {"title": str(row.title), "description": str(row.description),
            "cuisine": str(row.cuisine), "vegetarian": bool(row.vegetarian),
            "minutes": int(row.minutes), "embed_model": MODEL}

def build_recipes(client):
    if client.collection_exists("recipes"):
        client.delete_collection("recipes")
    client.create_collection("recipes", vectors_config=models.VectorParams(
        size=doc_emb.shape[1], distance=models.Distance.COSINE))
    client.upsert("recipes", points=[
        models.PointStruct(id=int(r.id), vector=doc_emb[i].tolist(), payload=to_payload(r))
        for i, r in enumerate(recipes.itertuples())])
    return client.count("recipes").count

quick = models.Filter(must=[models.FieldCondition(key="minutes", range=models.Range(lte=30))])

def ask(client, k=3):
    hits = client.query_points("recipes", query=warm_query, query_filter=quick, limit=k).points
    return [[h.payload["title"], round(h.score, 3)] for h in hits]

print("build_recipes(client) and ask(client) are ready")

build_recipes(client) and ask(client) are ready


### Step 2 · Mode 1: in memory
`":memory:"` keeps everything in RAM. Nothing touches the disk:

In [3]:
memory = QdrantClient(":memory:")
print("points:", build_recipes(memory))
answers = {"memory": ask(memory)}
print(answers["memory"])
memory.close()

points: 50
[['Gazpacho', 0.399], ['Lemon sorbet', 0.348], ['Hummus', 0.328]]


Close it and open a new `:memory:` client: the collection is gone. That is exactly what you want
for a unit test (a clean database every time) and exactly what you don't want for real data.

In [4]:
memory = QdrantClient(":memory:")
print("collections after reopening:", memory.get_collections().collections)
memory.close()

collections after reopening: []


### Step 3 · Mode 2: a folder on disk
`path=` is still local mode, but it saves to a folder (lesson 07). Build, close, and reopen with a
**new** client:

In [5]:
DB_PATH = "../data/cache/qdrant_13"
shutil.rmtree(DB_PATH, ignore_errors=True)               # start from an empty folder every run
folder = QdrantClient(path=DB_PATH)
print("points:", build_recipes(folder))
folder.close()

folder = QdrantClient(path=DB_PATH)                      # a fresh client, same folder
print("after reopening:", folder.count("recipes").count, "points")
answers["folder"] = ask(folder)
print(answers["folder"])
print("files:", sorted(os.listdir(f"{DB_PATH}/collection/recipes")))

points: 50
after reopening: 50 points
[['Gazpacho', 0.399], ['Lemon sorbet', 0.348], ['Hummus', 0.328]]
files: ['storage.sqlite']


The data survived. It lives in one SQLite file, and only **one** client may open the folder at a
time (lesson 07's `RuntimeError`). Close it before moving on:

In [6]:
folder.close()
print("folder released")

folder released


### Step 4 · Mode 3: a Qdrant server in Docker
The server is a separate program. Its settings live in `infra/compose/qdrant-single.yml`: image
`qdrant/qdrant:v1.19.2` (pinned), REST on port 6333, gRPC on 6334, a named volume for the data.
The next cells only run with Docker. Their results go into `server_log`; when they are skipped,
`from_server` reads the recorded output of a real run instead.

In [7]:
RECORDED = "../data/recorded/13.json"
server_log = {}                                          # filled by the needs-docker cells

def from_server(key):
    if key in server_log:
        return server_log[key], "live"
    with open(RECORDED) as f:
        return json.load(f)[key], "recorded output"

print("recorded keys:", sorted(json.load(open(RECORDED))) if os.path.exists(RECORDED) else "none")

recorded keys: ['after_restart', 'answers', 'quirks', 'two_clients', 'version']


Start the container, then **poll** the server's `/readyz` health endpoint until it answers (no
fixed sleep). The root URL `/` reports the version.

In [8]:
import socket, subprocess, time, urllib.request
QDRANT_URL = "http://localhost:6333"
COMPOSE = ["docker", "compose", "-f", "../infra/compose/qdrant-single.yml"]

def wait_ready(url, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            return urllib.request.urlopen(f"{url}/readyz", timeout=2).read().decode()
        except OSError:
            time.sleep(0.5)
    raise TimeoutError(f"{url} not ready after {timeout}s")

if socket.socket().connect_ex(("localhost", 6333)) == 0:
    raise RuntimeError("port 6333 is already in use; stop the other Qdrant first")
subprocess.run(COMPOSE + ["up", "-d"], check=True, capture_output=True)
print("readyz:", wait_ready(QDRANT_URL))
server_log["version"] = json.load(urllib.request.urlopen(QDRANT_URL))["version"]
print("server version:", server_log["version"])

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Now the only change in our code: `QdrantClient(url=...)` instead of `path=...`. Same functions,
same calls.

In [9]:
server = QdrantClient(url=QDRANT_URL)
print("points:", build_recipes(server))
server_log["answers"] = ask(server)
print(server_log["answers"])
info = server.get_collection("recipes")
print("status:", info.status, "| segments:", info.segments_count)

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Two things local mode cannot do. First, **restart** the server process: the data is on the
container's volume, so it comes back. Second, **two clients at once**: a second client (in real
life, another app or another machine) reads the same collection while the first stays open.

In [10]:
server.close()
subprocess.run(COMPOSE + ["restart"], check=True, capture_output=True)
print("readyz after restart:", wait_ready(QDRANT_URL))
server = QdrantClient(url=QDRANT_URL)
other_app = QdrantClient(url=QDRANT_URL)
server_log["after_restart"] = server.count("recipes").count
server_log["two_clients"] = [server.count("recipes").count, other_app.count("recipes").count]
print("points after restart:", server_log["after_restart"])
print("client 1 sees", server_log["two_clients"][0], "| client 2 sees", server_log["two_clients"][1])
other_app.close()

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


If Docker was skipped, here is what those cells printed on a real run:

In [11]:
for key in ["version", "answers", "after_restart", "two_clients"]:
    value, source = from_server(key)
    print(f"{key:<14} {value}   ({source})")

version        1.19.2   (recorded output)
answers        [['Gazpacho', 0.399], ['Lemon sorbet', 0.348], ['Hummus', 0.328]]   (recorded output)
after_restart  50   (recorded output)
two_clients    [50, 50]   (recorded output)


### Step 5 · Mode 4: Qdrant Edge
Qdrant Edge (`qdrant-edge-py`, beta) is the server's storage and search engine as an in-process
library: no network, no Docker, but real segments and real indexes. Its API is **not**
`QdrantClient`. You create an `EdgeShard` in a folder from an `EdgeConfig`, and write with
`UpdateOperation`s:

In [12]:
from qdrant_edge import (CountRequest, Distance, EdgeConfig, EdgeShard, EdgeVectorParams,
                         FieldCondition, Filter, MatchValue, Point, Query, QueryRequest,
                         RangeFloat, UpdateOperation)

EDGE_PATH = "../data/cache/edge_13"
shutil.rmtree(EDGE_PATH, ignore_errors=True)
os.makedirs(EDGE_PATH)
shard = EdgeShard.create(EDGE_PATH, EdgeConfig(
    vectors=EdgeVectorParams(size=doc_emb.shape[1], distance=Distance.Cosine)))
shard.update(UpdateOperation.upsert_points([
    Point(id=int(r.id), vector=doc_emb[i].tolist(), payload=to_payload(r))
    for i, r in enumerate(recipes.itertuples())]))
print("points:", shard.count(CountRequest()))

points: 50


The same quick-recipe search, in Edge's words: a `QueryRequest` holding a `Query.Nearest`. Filters
use the same names as before (`Filter`, `FieldCondition`), imported from `qdrant_edge`; the range
class is `RangeFloat`.

In [13]:
def edge_ask(shard, k=3):
    hits = shard.query(QueryRequest(
        query=Query.Nearest(warm_query), limit=k, with_payload=True,
        filter=Filter(must=[FieldCondition(key="minutes", range=RangeFloat(lte=30))])))
    return [[h.payload["title"], round(h.score, 3)] for h in hits]

answers["edge"] = edge_ask(shard)
print(answers["edge"])

[['Gazpacho', 0.399], ['Lemon sorbet', 0.348], ['Hummus', 0.328]]


Why a minutes filter and not lesson 11's `vegetarian` one? Edge is beta, and in `qdrant-edge-py`
0.8.0 a boolean `MatchValue` matches nothing (it is passed on as the number 1). See it:

In [14]:
veg_edge = Filter(must=[FieldCondition(key="vegetarian", match=MatchValue(value=True))])
print("vegetarian recipes in the CSV:", int(recipes.vegetarian.sum()))
print("Edge count with MatchValue(True):", shard.count(CountRequest(filter=veg_edge)))

vegetarian recipes in the CSV: 33
Edge count with MatchValue(True): 0


Edge saves to its folder. `close()` it, then reopen with `EdgeShard.load` (not `create`):

In [15]:
shard.close()
shard = EdgeShard.load(EDGE_PATH)
print("after reload:", shard.count(CountRequest()), "points |", edge_ask(shard)[0])
print("folder:", sorted(os.listdir(EDGE_PATH)))

after reload: 50 points | ['Gazpacho', 0.399]
folder: ['edge_config.json', 'segments', 'wal']


### Step 6 · Same question, four modes
Put the four answers side by side (the server column is live or recorded, as labelled):

In [16]:
answers["server"], source = from_server("answers")
for rank in range(3):
    row = [answers[m][rank] for m in ["memory", "folder", "server", "edge"]]
    print(f"#{rank + 1}", " | ".join(f"{t} {s:.3f}" for t, s in row))
same = all(answers[m] == answers["memory"] for m in answers)
print("identical in all four modes:", same, f"(server: {source})")

#1 Gazpacho 0.399 | Gazpacho 0.399 | Gazpacho 0.399 | Gazpacho 0.399
#2 Lemon sorbet 0.348 | Lemon sorbet 0.348 | Lemon sorbet 0.348 | Lemon sorbet 0.348
#3 Hummus 0.328 | Hummus 0.328 | Hummus 0.328 | Hummus 0.328
identical in all four modes: True (server: recorded output)


Same titles, same scores, in every mode. (Odd answers for a warm query: MiniLM latches on to
'cold' and 'day', and the 30-minute filter removes the stews. This lesson cares only that all
modes agree.) With 50 recipes every mode compares the query with every
vector, so they must agree. The differences appear when there is an **index**.

### Step 7 · Local mode ignores search settings
An HNSW index (lesson 29 explains it) trades a little accuracy for speed, tuned per query with
`search_params=SearchParams(hnsw_ef=...)`: bigger `hnsw_ef` = more careful = closer to exact. To
see it we need more than 50 vectors, so we make 5,000 random 64-number vectors (a stand-in for a
big menu) and the exact top 10 for 50 random queries. **Recall** = the share of the true top 10
a search returns.

In [17]:
rng = np.random.default_rng(13)
N, D = 5000, 64
big = rng.normal(size=(N, D)).astype(np.float32)
big /= np.linalg.norm(big, axis=1, keepdims=True)
big_queries = rng.normal(size=(50, D)).astype(np.float32)
big_queries /= np.linalg.norm(big_queries, axis=1, keepdims=True)
true_top10 = np.argsort(-(big_queries @ big.T), axis=1)[:, :10]

def recall(found):
    return round(float(np.mean([len(set(f) & set(t)) / 10 for f, t in zip(found, true_top10)])), 2)

print("vectors:", big.shape, "| queries:", big_queries.shape)

vectors: (5000, 64) | queries: (50, 64)


Local mode first. We try three `hnsw_ef` values, down to a deliberately careless 16, plus
`exact=True`, which skips any index:

In [18]:
SETTINGS = {"hnsw_ef=16": dict(hnsw_ef=16), "hnsw_ef=64": dict(hnsw_ef=64),
            "hnsw_ef=256": dict(hnsw_ef=256), "exact=True": dict(exact=True)}

local = QdrantClient(":memory:")
local.create_collection("synthetic", vectors_config=models.VectorParams(
    size=D, distance=models.Distance.COSINE))
local.upsert("synthetic", points=models.Batch(ids=list(range(N)), vectors=big.tolist()))

def local_recall(**params):
    return recall([[p.id for p in local.query_points("synthetic", query=q.tolist(), limit=10,
                    search_params=models.SearchParams(**params)).points] for q in big_queries])

recalls = {"local": {name: local_recall(**params) for name, params in SETTINGS.items()}}
print("local mode recall:", recalls["local"])
print("indexed vectors:", local.get_collection("synthetic").indexed_vectors_count)
local.close()

/var/tmp/ipykernel_29673/2458163962.py:10: UserWarning: Local mode performs exact (brute-force) search, so `search_params` has no effect, with the exception of `idf`.
  return recall([[p.id for p in local.query_points("synthetic", query=q.tolist(), limit=10,


local mode recall: {'hnsw_ef=16': 1.0, 'hnsw_ef=64': 1.0, 'hnsw_ef=256': 1.0, 'exact=True': 1.0}
indexed vectors: 0


Recall 1.0 every time, and a `UserWarning`: "Local mode performs exact (brute-force) search, so
`search_params` has no effect". No index was built (0 indexed vectors). A test that passes in
local mode tells you nothing about how your tuning behaves in production.

Now Edge, with a real HNSW index. A segment this small is normally just scanned, so two settings
(both in kilobytes) force an index: `indexing_threshold=10` and `full_scan_threshold=10`. Edge has
no background worker: you call `optimize()` to build the index. Edge's `HnswIndexConfig` requires
`m` and `ef_construct`; 16 and 100 are the server's defaults, and lesson 29 explains them.

In [19]:
from qdrant_edge import EdgeOptimizersConfig, HnswIndexConfig, SearchParams

shutil.rmtree("../data/cache/edge_13_synthetic", ignore_errors=True)
os.makedirs("../data/cache/edge_13_synthetic")
big_shard = EdgeShard.create("../data/cache/edge_13_synthetic", EdgeConfig(
    vectors=EdgeVectorParams(size=D, distance=Distance.Cosine),
    hnsw_config=HnswIndexConfig(m=16, ef_construct=100, full_scan_threshold=10),
    optimizers=EdgeOptimizersConfig(indexing_threshold=10)))
big_shard.update(UpdateOperation.upsert_points([Point(id=i, vector=big[i].tolist()) for i in range(N)]))
big_shard.optimize()
print(big_shard.info())

ShardInfo(segments_count=2, points_count=5000, indexed_vectors_count=5000, payload_schema={})


All 5,000 vectors are indexed. Now the same four settings on Edge:

In [20]:
def edge_recall(**params):
    return recall([[p.id for p in big_shard.query(QueryRequest(
        query=Query.Nearest(q.tolist()), limit=10, params=SearchParams(**params)))]
        for q in big_queries])

recalls["edge"] = {name: edge_recall(**params) for name, params in SETTINGS.items()}
big_shard.close()

print(f"{'search_params':<14}{'local mode':>11}{'Edge':>7}")
for name in SETTINGS:
    print(f"{name:<14}{recalls['local'][name]:>11}{recalls['edge'][name]:>7}")

search_params  local mode   Edge
hnsw_ef=16            1.0   0.58
hnsw_ef=64            1.0   0.94
hnsw_ef=256           1.0    1.0
exact=True            1.0    1.0


On Edge, `hnsw_ef` matters: recall rises from about 0.6 at `hnsw_ef=16` to 1.0 at `hnsw_ef=256`.
Graph building uses randomness, so these numbers move by up to about 0.03 between runs (0.58-0.64
at hnsw_ef=16 in our runs). Local mode says 1.0 for everything. Tune and benchmark on a real
engine (a server or Edge), never on local mode.

### Step 8 · Small behaviours that differ
Local mode reimplements Qdrant in Python, so a few edge cases differ. Here is one that affects
lesson 12's `update_mode`: what happens to an id you just deleted. `quirks` deletes a point, then
writes it back with `update_only` ("only if it exists").

In [21]:
def quirks(client):
    build_recipes(client)
    point = models.PointStruct(id=50, vector=doc_emb[0].tolist(), payload={"title": "Test"})
    client.delete("recipes", points_selector=[50])
    client.upsert("recipes", points=[point], update_mode=models.UpdateMode.UPDATE_ONLY)
    return {"update_only re-creates deleted id 50": bool(client.retrieve("recipes", ids=[50]))}

local = QdrantClient(":memory:")
local_quirks = quirks(local)
print(local_quirks)

{'update_only re-creates deleted id 50': True}


On the server:

In [22]:
server_log["quirks"] = quirks(server)
print(server_log["quirks"])

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Side by side:

In [23]:
server_quirks, source = from_server("quirks")
for check, value in local_quirks.items():
    flag = "" if value == server_quirks[check] else "   <- differs"
    print(f"{check:<38} local {str(value):<6} server {str(server_quirks[check]):<6}{flag}")
print(f"(server column: {source})")
local.close()

update_only re-creates deleted id 50   local True   server False    <- differs
(server column: recorded output)


On the server a deleted id is simply gone, so `update_only` skips it. Local mode remembers the
deleted id for the rest of the session and re-creates it. Rule: write tests in local mode, but
check write logic and anything about speed or tuning on a real server.

### Step 9 · One switch to change modes
Real code should not hard-code the mode. Read it from the environment: if `QDRANT_URL` is set,
talk to that server, otherwise use a local folder. Tests, laptops and production then run the same
code with a different setting.

In [24]:
def connect(default_path="../data/cache/qdrant_13"):
    url = os.environ.get("QDRANT_URL")
    if url:
        return QdrantClient(url=url), f"server at {url}"
    return QdrantClient(path=default_path), f"local folder {default_path}"

client, where = connect()
print("connected to:", where)
print("top hit:", ask(client)[0])
client.close()

connected to: local folder ../data/cache/qdrant_13
top hit: ['Gazpacho', 0.399]


With `QDRANT_URL=http://localhost:6333` set before starting Jupyter, the same cell would use the
server. Choosing between the four modes:

| Situation | Use | Why |
| --- | --- | --- |
| Unit tests, CI, a quick experiment | `QdrantClient(":memory:")` | clean every time, no setup |
| Learning, a small personal tool | `QdrantClient(path=...)` | saved to disk, one process only |
| An app on one device, offline, real indexes | Qdrant Edge | real engine in-process, call `optimize()` yourself |
| A shared service, many users or apps, tuning | Qdrant server | real indexes, background indexing, many clients |

## What just happened
- Steps 2-3: the same `build_recipes` and `ask` ran in memory (lost on close) and in a folder (50
  points after reopening).
- Step 4: one changed line pointed them at a Docker server, which kept 50 points across a
  restart and served two clients at once.
- Steps 5-6: Edge needed its own API, and all four modes returned the same top 3 with the same
  scores.
- Step 7: local vs Edge on 5,000 vectors: local mode returned recall 1.0 for any `hnsw_ef` (and
  warned it ignores `search_params`); Edge rose from about 0.6 to 1.0 as `hnsw_ef` grew.
- Step 8: `update_only` re-creates a just-deleted id in local mode but not on the server.

## Common mistakes

**1. Creating an Edge shard in a folder that already has one.** `EdgeShard.create` is for a new,
empty folder only.

In [25]:
try:
    shard.close()
    shard = EdgeShard.create(EDGE_PATH, EdgeConfig(
        vectors=EdgeVectorParams(size=doc_emb.shape[1], distance=Distance.Cosine)))
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

Exception: Service runtime error: cannot create edge shard: path already contains segment data


It prints `Exception: Service runtime error: cannot create edge shard: path already contains
segment data`. Fix: open an existing shard with `EdgeShard.load(path)`:

In [26]:
shard = EdgeShard.load(EDGE_PATH)
print("loaded:", shard.count(CountRequest()), "points")

loaded: 50 points


**2. Pointing the client at a server that is not running.** Creating a `QdrantClient(url=...)`
does not fail, so a wrong port or a stopped container only shows up at the first call. Port 6399
has no server. `check_compatibility=False` turns off the client's version check against the
server. Without it you would also see `UserWarning: Failed to obtain server version` before the
same error.

In [27]:
try:
    nowhere = QdrantClient(url="http://localhost:6399", check_compatibility=False)  # skip the version check (see above)
    nowhere.get_collections()
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ResponseHandlingException: [Errno 111] Connection refused


It prints `ResponseHandlingException: [Errno 111] Connection refused`. Fix: start the server,
check the URL and port, and poll `/readyz` before the first request, as `wait_ready` does in
Step 4.

## Try it
1. **Edge keeps deletes.** Gazpacho comes off the menu. Delete id 50 from the Edge `shard` with
   `UpdateOperation.delete_points([50])`, close the shard, load it again, and print the count and
   the top 3 of `edge_ask` (Gazpacho, 10 minutes, is a quick recipe).
2. **How careful is careful enough?** On the 5,000-vector Edge shard (rebuild it as in Step 7),
   print recall for `hnsw_ef` in [10, 20, 40, 80, 160] and find the smallest value that reaches
   0.9.

In [28]:
# your code here

Clean up: close the Edge shard and the server client, save what the server printed (this
refreshes the recorded output used when Docker is skipped), and stop the container. `down -v`
also deletes the server's data volume.

In [29]:
shard.close()
print("Edge shard closed")

Edge shard closed


The Docker part of the clean-up:

In [30]:
server.close()
with open(RECORDED, "w") as f:
    json.dump(server_log, f, indent=1)
subprocess.run(COMPOSE + ["down", "-v"], check=True, capture_output=True)
print("saved", sorted(server_log), "to", RECORDED, "| server stopped")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


## Key terms
- **local mode** — qdrant-client's built-in Python Qdrant (`":memory:"` or `path=`): exact
  search, no indexes, ignores `search_params`; for development and tests.
- **in-memory client** — `QdrantClient(":memory:")`: local mode in RAM; everything is lost when the
  client closes.
- **client-server** — the database is its own program (here in Docker); clients connect over
  the network with `QdrantClient(url=...)` (REST 6333, gRPC 6334), many at once.
- **persistence** — data that survives closing the client or restarting the process (folder,
  server volume, Edge folder; not `:memory:`).
- **Qdrant Edge** — `qdrant-edge-py`: the Qdrant engine as an in-process library (`EdgeShard`);
  real indexes, no network, no background optimizer, one shard.
- **readiness check** — polling a health endpoint (Qdrant: `/readyz`) until the server is ready.
- **search params** — per-query search settings such as `hnsw_ef` and `exact`; real engines use
  them, local mode ignores them.
- **recall** — the share of the true top-k results that a search returns (1.0 = all of them).